In [ ]:
# Run this first — takes about 30 seconds
!pip install pandas numpy matplotlib seaborn plotly -q
print("✅ Libraries installed!")

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

localities = [
    "Andheri West", "Andheri East", "Bandra West", "Bandra East",
    "Borivali West", "Borivali East", "Kandivali West", "Kandivali East",
    "Malad West", "Malad East", "Goregaon West", "Goregaon East",
    "Powai", "Vikhroli", "Mulund West", "Mulund East",
    "Thane West", "Thane East", "Navi Mumbai", "Kharghar",
    "Panvel", "Airoli", "Nerul", "Vashi",
    "Dadar", "Parel", "Worli", "Lower Parel",
    "Kurla", "Chembur", "Ghatkopar West", "Ghatkopar East",
    "Dombivli", "Kalyan", "Ulhasnagar", "Mira Road",
    "Vasai", "Virar", "Nalasopara", "Dahisar"
]

locality_price_map = {
    "Bandra West":    (250, 900),  "Worli":          (300, 1200),
    "Lower Parel":    (200, 800),  "Parel":           (180, 700),
    "Dadar":          (150, 600),  "Powai":           (120, 500),
    "Andheri West":   (100, 400),  "Andheri East":    (80,  350),
    "Goregaon West":  (80,  300),  "Goregaon East":   (70,  280),
    "Malad West":     (70,  250),  "Malad East":      (60,  220),
    "Kandivali West": (65,  230),  "Kandivali East":  (55,  200),
    "Borivali West":  (60,  210),  "Borivali East":   (50,  190),
    "Thane West":     (55,  200),  "Thane East":      (45,  180),
    "Navi Mumbai":    (50,  220),  "Kharghar":        (45,  180),
    "Vashi":          (70,  260),  "Nerul":           (60,  220),
    "Airoli":         (50,  180),  "Panvel":          (35,  150),
    "Kurla":          (65,  240),  "Chembur":         (80,  300),
    "Ghatkopar West": (75,  280),  "Ghatkopar East":  (65,  250),
    "Vikhroli":       (60,  220),  "Mulund West":     (70,  260),
    "Mulund East":    (60,  230),  "Bandra East":     (150, 550),
    "Dombivli":       (30,  120),  "Kalyan":          (28,  110),
    "Mira Road":      (40,  160),  "Vasai":           (25,  100),
    "Virar":          (20,   90),  "Nalasopara":      (18,   85),
    "Dahisar":        (45,  170),  "Ulhasnagar":      (20,   80),
}

rows = []
for _ in range(n):
    locality  = np.random.choice(localities)
    low, high = locality_price_map.get(locality, (40, 200))
    bhk       = np.random.choice([1, 1, 2, 2, 2, 3, 3, 3, 4, 5])
    price     = round(np.random.uniform(low * bhk * 0.4, high * bhk * 0.25), 2)
    area      = round(np.random.uniform(400 + bhk * 150, 600 + bhk * 300), 0)
    rows.append({
        "locality":           locality,
        "bhk":                bhk,
        "price_lakhs":        price,
        "area_sqft":          area,
        "furnishing_status":  np.random.choice(["Furnished", "Semi-Furnished", "Unfurnished"]),
        "facing":             np.random.choice(["East", "West", "North", "South"]),
        "status":             np.random.choice(["Ready to Move", "Under Construction"]),
        "floor":              np.random.randint(1, 30),
        "parking":            np.random.choice([0, 1, 2]),
        "bathroom":           np.random.choice([1, 2, 2, 3, 3]),
        "city":               "Mumbai",
    })

df_raw = pd.DataFrame(rows)
print(f"✅ Dataset created: {len(df_raw):,} rows × {len(df_raw.columns)} columns")
print(f"   Price range : ₹{df_raw['price_lakhs'].min():.1f}L – ₹{df_raw['price_lakhs'].max():.1f}L")
df_raw.head()

In [ ]:
# Save raw data as Bronze layer
df_raw.to_csv("bronze_mumbai_raw.csv", index=False)
print("✅ Bronze layer saved → bronze_mumbai_raw.csv")
print(f"   Shape  : {df_raw.shape}")
print(f"   Nulls  : {df_raw.isnull().sum().sum()}")
df_raw.describe()

In [ ]:
# Load from Bronze and clean
df = pd.read_csv("bronze_mumbai_raw.csv")

before = len(df)

# Remove bad rows
df = df.dropna(subset=["price_lakhs", "area_sqft", "locality", "bhk"])
df = df[(df["price_lakhs"] > 0) & (df["area_sqft"] > 0)]

# Add calculated columns
df["price_per_sqft"]     = (df["price_lakhs"] * 100000 / df["area_sqft"]).round(2)

df["price_segment"] = pd.cut(
    df["price_lakhs"],
    bins   = [0, 50, 100, 200, float("inf")],
    labels = ["Affordable (<50L)", "Mid-Range (50-100L)", "Premium (1-2Cr)", "Luxury (2Cr+)"]
).astype(str)

df["affordability_score"] = (
    100 - ((df["price_lakhs"] - df["price_lakhs"].min()) /
           (df["price_lakhs"].max() - df["price_lakhs"].min()) * 100)
).round(1)

# Save Silver layer
df.to_csv("silver_mumbai_clean.csv", index=False)

print(f"✅ Silver layer saved → silver_mumbai_clean.csv")
print(f"   Rows before clean : {before:,}")
print(f"   Rows after clean  : {len(df):,}")
print(f"   New columns added : price_per_sqft, price_segment, affordability_score")
df.head()

In [ ]:
# Gold Table 1 — Locality Summary
gold_locality = (
    df.groupby("locality")
    .agg(
        total_listings    = ("price_lakhs",      "count"),
        avg_price_lakhs   = ("price_lakhs",      "mean"),
        min_price_lakhs   = ("price_lakhs",      "min"),
        max_price_lakhs   = ("price_lakhs",      "max"),
        avg_area_sqft     = ("area_sqft",         "mean"),
        avg_price_per_sqft= ("price_per_sqft",    "mean"),
        avg_afford_score  = ("affordability_score","mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("avg_price_lakhs")
)

# Gold Table 2 — BHK Analysis
gold_bhk = (
    df.groupby("bhk")
    .agg(
        total_listings    = ("price_lakhs", "count"),
        avg_price_lakhs   = ("price_lakhs", "mean"),
        avg_area_sqft     = ("area_sqft",    "mean"),
        avg_price_per_sqft= ("price_per_sqft","mean"),
    )
    .round(2)
    .reset_index()
)

# Gold Table 3 — Price Segment Distribution
gold_segment = (
    df.groupby("price_segment")
    .agg(
        total_listings  = ("price_lakhs", "count"),
        avg_price_lakhs = ("price_lakhs", "mean"),
        avg_area_sqft   = ("area_sqft",    "mean"),
    )
    .round(2)
    .reset_index()
)

# Gold Table 4 — Top 10 Affordable Localities
gold_top10_affordable = gold_locality.head(10)[
    ["locality", "avg_price_lakhs", "avg_price_per_sqft",
     "total_listings", "avg_afford_score"]
]

# Gold Table 5 — Furnishing Analysis
gold_furnishing = (
    df.groupby("furnishing_status")
    .agg(
        total_listings  = ("price_lakhs", "count"),
        avg_price_lakhs = ("price_lakhs", "mean"),
        avg_area_sqft   = ("area_sqft",    "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("avg_price_lakhs", ascending=False)
)

# Save all Gold tables
gold_locality.to_csv("gold_locality_summary.csv",         index=False)
gold_bhk.to_csv("gold_bhk_analysis.csv",                  index=False)
gold_segment.to_csv("gold_segment_distribution.csv",      index=False)
gold_top10_affordable.to_csv("gold_top10_affordable.csv", index=False)
gold_furnishing.to_csv("gold_furnishing_analysis.csv",    index=False)

print("✅ Gold layer created!")
print("   → gold_locality_summary.csv")
print("   → gold_bhk_analysis.csv")
print("   → gold_segment_distribution.csv")
print("   → gold_top10_affordable.csv")
print("   → gold_furnishing_analysis.csv")
print("\n📋 Top 10 Affordable Localities in Mumbai:")
print(gold_top10_affordable.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 14))
fig.suptitle("Mumbai Real Estate — Data Engineering Dashboard",
             fontsize=18, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

# ── Chart 1: Top 15 localities by avg price ──────────────
ax1 = fig.add_subplot(gs[0, :2])
top15 = gold_locality.tail(15)
colors = ["#e74c3c" if p > 200 else "#f39c12" if p > 80 else "#2ecc71"
          for p in top15["avg_price_lakhs"]]
bars = ax1.barh(top15["locality"], top15["avg_price_lakhs"], color=colors)
ax1.set_xlabel("Avg Price (₹ Lakhs)")
ax1.set_title("Top 15 Localities by Average Price", fontweight="bold")
for bar, val in zip(bars, top15["avg_price_lakhs"]):
    ax1.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             f"₹{val:.0f}L", va="center", fontsize=8)
ax1.legend(handles=[
    plt.Rectangle((0,0),1,1, color="#e74c3c", label="Luxury >200L"),
    plt.Rectangle((0,0),1,1, color="#f39c12", label="Premium 80-200L"),
    plt.Rectangle((0,0),1,1, color="#2ecc71", label="Affordable <80L"),
], fontsize=8, loc="lower right")

# ── Chart 2: Price segment pie ────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.pie(gold_segment["total_listings"],
        labels=gold_segment["price_segment"],
        autopct="%1.0f%%",
        colors=["#2ecc71","#f39c12","#e67e22","#e74c3c"],
        startangle=90, textprops={"fontsize": 8})
ax2.set_title("Price Segment Split", fontweight="bold")

# ── Chart 3: BHK vs avg price bar ─────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(gold_bhk["bhk"].astype(str) + " BHK",
        gold_bhk["avg_price_lakhs"],
        color=["#3498db","#2ecc71","#f39c12","#e74c3c","#9b59b6"])
ax3.set_title("Avg Price by BHK Type", fontweight="bold")
ax3.set_ylabel("Avg Price (₹ Lakhs)")
ax3.set_xlabel("BHK Type")
for i, v in enumerate(gold_bhk["avg_price_lakhs"]):
    ax3.text(i, v + 1, f"₹{v:.0f}L", ha="center", fontsize=8)

# ── Chart 4: Furnishing status ────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(gold_furnishing["furnishing_status"],
        gold_furnishing["avg_price_lakhs"],
        color=["#e74c3c","#f39c12","#2ecc71"])
ax4.set_title("Avg Price by Furnishing", fontweight="bold")
ax4.set_ylabel("Avg Price (₹ Lakhs)")
for i, v in enumerate(gold_furnishing["avg_price_lakhs"]):
    ax4.text(i, v + 1, f"₹{v:.0f}L", ha="center", fontsize=8)

# ── Chart 5: Top 10 affordable localities ─────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.barh(gold_top10_affordable["locality"],
         gold_top10_affordable["avg_price_lakhs"],
         color="#2ecc71")
ax5.set_title("Top 10 Affordable Areas", fontweight="bold")
ax5.set_xlabel("Avg Price (₹ Lakhs)")
ax5.invert_yaxis()

# ── Chart 6: Price vs Area scatter ───────────────────────
ax6 = fig.add_subplot(gs[2, :2])
scatter_colors = df["bhk"].map({1:"#3498db",2:"#2ecc71",3:"#f39c12",4:"#e74c3c",5:"#9b59b6"})
ax6.scatter(df["area_sqft"], df["price_lakhs"],
            c=scatter_colors, alpha=0.4, s=15)
ax6.set_xlabel("Area (sqft)")
ax6.set_ylabel("Price (₹ Lakhs)")
ax6.set_title("Price vs Area (colored by BHK)", fontweight="bold")
ax6.legend(handles=[
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor=c, markersize=8, label=f"{k} BHK")
    for k, c in {1:"#3498db",2:"#2ecc71",3:"#f39c12",4:"#e74c3c",5:"#9b59b6"}.items()
], fontsize=8)

#  Affordability score by locality (top 10) ────
ax7 = fig.add_subplot(gs[2, 2])
top10_afford = gold_locality.nlargest(10, "avg_afford_score")
ax7.barh(top10_afford["locality"], top10_afford["avg_afford_score"], color="#3498db")
ax7.set_title("Top 10 by Affordability Score", fontweight="bold")
ax7.set_xlabel("Score (higher = more affordable)")
ax7.invert_yaxis()

plt.savefig("mumbai_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved → mumbai_dashboard.png")

In [ ]:
# Install libraries (only once)

!pip install scikit-learn -q

In [ ]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# Features and target

X = df.drop("price_segment", axis=1)
y = df["price_segment"]

# Convert categorical columns to numeric

X = pd.get_dummies(X, drop_first=True)

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Models

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

# Training and evaluation

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{name}")
    print("-" * 40)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
from google.colab import files

# Download all output files to your computer
files.download("bronze_mumbai_raw.csv")
files.download("silver_mumbai_clean.csv")
files.download("gold_locality_summary.csv")
files.download("gold_bhk_analysis.csv")
files.download("gold_segment_distribution.csv")
files.download("gold_top10_affordable.csv")
files.download("gold_furnishing_analysis.csv")
files.download("mumbai_dashboard.png")

print("✅ All files downloaded!")

In [ ]:
files.download("mumbai_dashboard.png")

In [ ]:
files.download("gold_furnishing_analysis.csv")